In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
      print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

import time
!pip install 'kaggle-environments==0.1.6'
import math
import copy
import random

from kaggle_environments import evaluate, make, utils

env = make("connectx", debug=True)
# env.render() # text format

board_to_node_map = {}

class Node:
    def __init__ (self, board, parent, to_play) :
        self.board = board
        self.parent = parent
        self.to_play = to_play
        self.t = 0
        self.n = 0
        self.children = []
        self.untried_moves = [c for c in range(n_columns) if board[c] == 0]

def get_node (board, parent, board_to_node_map):
    if tuple(board) in board_to_node_map:
        if board_to_node_map[tuple(board)] == root_node: print("Using cached root node")
        return board_to_node_map[tuple(board)]

    to_play = 1 if board.count(1)==board.count(2) else 2
    node = Node(board, parent, to_play)
    if node.parent and node.parent.parent == root_node: board_to_node_map[tuple(board)] = node
    return node

def UCB (node) :
    """Calculates UCB value of a node"""
    return (node.t/node.n) + 1.414 * (math.log(node.parent.n) / node.n)**0.5

def choose_max_UCB_child (node):
    """Returns child node with highest UCB score"""
    max_UCB = float('-inf')
    max_UCB_child_node = None
    for child_node in node.children:
        if child_node.n == 0: return child_node
        child_ucb = UCB(child_node)
        if child_ucb > max_UCB:
            max_UCB, max_UCB_child_node = child_ucb, child_node

    return max_UCB_child_node

def terminalStatus (board):
    """Returns two elements: isTerminal, and the mark of the winner if it is terminal"""
    for r in range(n_rows):
        for c in range(n_columns):
            mark = board[r * n_columns + c]
            
            if mark == 0: continue
            # Check Horizontal
            if c + inarow <= n_columns and all(board[r * n_columns + c + i] == mark for i in range(inarow)): return [True, mark]
            # Check Vertical
            if r + inarow <= n_rows and all(board[(r + i) * n_columns + c] == mark for i in range(inarow)): return [True, mark]
            # Check Diagonal
            if r + inarow <= n_rows and c + inarow <= n_columns and all(board[(r + i) * n_columns + c + i] == mark for i in range(inarow)): return [True, mark]
            # Check Anti-Diagonal
            if r + inarow <= n_rows and c - inarow + 1 >= 0 and all(board[(r + i) * n_columns + c - i] == mark for i in range(inarow)): return [True, mark]
                
    return [True, 0] if 0 not in board[:n_columns] else [False, 0]

def playMove (board, col, player_mark):
    new_board = list(board)

    for row in range(n_rows):
        if row==n_rows-1 or new_board[n_columns*(row+1) + col] != 0:
            new_board[n_columns*row + col] = player_mark
            break
    
    return new_board

def rollout (node):
    """
    Performs rollout from the input node.
    Returns the mark of the winner found at the end of the rollout.
    """
    
    current_board = node.board
    current_to_play = node.to_play

    while not terminalStatus(current_board)[0]:
        valid_cols = [c for c in range(n_columns) if current_board[c] == 0]
        
        if not valid_cols: break
            
        col = random.choice(valid_cols)
        current_board = playMove(current_board, col, current_to_play)
        current_to_play = 3 - current_to_play

    return terminalStatus(current_board)[1]

def backpropagate_vals (node, winner_mark):
    """
    Adds val to `t` values of all ancestors in the lineage
    Adds 1 to `n` values of all ancestors in the lineage
    """
    while node is not None:
        if winner_mark == 3-node.to_play: node.t = node.t + 1
        elif winner_mark == 0: node.t = node.t + 0.5
        node.n = node.n + 1
        node = node.parent

def pick_best_move (node):
    """
    Finds child that was visited most number of times
    """
    board = node.board
    column = -1

    most_visits = -1
    best_child = node.children[0]
    
    for child_node in node.children:
        if child_node.n > most_visits:
            most_visits, best_child = child_node.n, child_node
            # print("Best move updated")
        
    diff = np.where(np.array(node.board) != np.array(best_child.board))[0]
    column = int(diff[0]) % n_columns
    
    if column == -1:
        print("hmmm")
        for idx in range(n_columns):
            if board[idx] == 0:
                column = idx
                break

    return column

def chooseMove (root_board, start_time, board_to_node_map) :
    global root_node
    if tuple(root_board) in board_to_node_map: 
        root_node = board_to_node_map[tuple(root_board)]
        # print("    Used cached root node")
        for child in root_node.children:
            for gc in child.children:
                board_to_node_map[tuple(gc.board)] = gc
                # print("        cached root's grandchild")
    else: 
        root_node = Node(root_board, None, 1 if root_board.count(1)==root_board.count(2) else 2)
        root_node.n = 1
        
    n_iters = 0

    while time.perf_counter()-start_time < 1.9: 
        n_iters = n_iters + 1
        ## print("MC iteration")
        current_node = root_node

        # is current_node a leaf?
        while not current_node.untried_moves and current_node.children :
            ## print("curr node is leaf")
            # yes
            current_node = choose_max_UCB_child(current_node) # Use UCB to go to next best child
            ## print("curr node updated")

        # current_node is not a leaf now

        # if current_node.n != 0: # has this node been visited before?
            ## print("curr node visited but no children")
            # yes
            # but it doesn't have any children
            # So add its children to the tree, each corresponding to a valid move on current_node
        
        if time.perf_counter()-start_time < 1.9: break

        if current_node.untried_moves:
            col = current_node.untried_moves.pop(random.randint(0, len(current_node.untried_moves)-1))
            child_board = playMove(current_node.board, col, current_node.to_play)
            child_node = Node(child_board, current_node, 3-current_node.to_play)
            
            if current_node.parent == root_node: 
                board_to_node_map[tuple(child_board)] = child_node
                # print("        cached root's grandchild")
            
            current_node.children.append(child_node)
            current_node = child_node
            
            ## print("children added to curr node")
            ## print(len(current_node.children))
                
            ## print("now looking at child[0]")
        
            if time.perf_counter()-start_time < 1.9: break

        # current_node has never been visited before - guaranteed
        # perform rollout
        ## print("Going to perform rollout")
        winner_mark = rollout(current_node)
        ## print("performed rollout")
        backpropagate_vals(current_node, winner_mark)
        ## print("performed backprop")

    '''
    print(f"DEBUG: Root node has {len(root_node.children)} children.")
    for i, child in enumerate(root_node.children):
        print(f"Child {i}: visits={child.n}, score={child.t}")
    '''

    # print(f"n_iters = {n_iters}")

    final_move_column = pick_best_move(root_node)
    ## print(f"Chosen final move column = {final_move_column}")
    return final_move_column

def my_agent (observation, configuration) :
    # print("Move triggered")
    start_time = time.perf_counter()
    # Keep track of the time at which the bot was prompted
    
    global n_columns, n_rows, inarow, player_mark, opp_mark, board_to_node_map

    n_columns = configuration.columns
    n_rows = configuration.rows
    inarow = configuration.inarow

    root_board = observation.board
    player_mark = observation.mark
    opp_mark = 3 - player_mark

    move = chooseMove(root_board, start_time, board_to_node_map)
    # move simply denotes the column in which to play the move

    if time.perf_counter() - start_time >= 2: print(">2s TIMEOUT")

    return move

'''
env.reset()
# Play as the first agent against default "random" agent.
env.run([my_agent, "random"])
env.render(mode="ipython", width=500, height=450)
'''

/kaggle/input/competitions/connectx/kaggle-environments-0.1.4/CONTRIBUTING.md
/kaggle/input/competitions/connectx/kaggle-environments-0.1.4/LICENSE
/kaggle/input/competitions/connectx/kaggle-environments-0.1.4/.gitignore
/kaggle/input/competitions/connectx/kaggle-environments-0.1.4/main.py
/kaggle/input/competitions/connectx/kaggle-environments-0.1.4/README.md
/kaggle/input/competitions/connectx/kaggle-environments-0.1.4/MANIFEST.in
/kaggle/input/competitions/connectx/kaggle-environments-0.1.4/requirements.txt
/kaggle/input/competitions/connectx/kaggle-environments-0.1.4/.gcloudignore
/kaggle/input/competitions/connectx/kaggle-environments-0.1.4/setup.py
/kaggle/input/competitions/connectx/kaggle-environments-0.1.4/kaggle_environments/schemas.json
/kaggle/input/competitions/connectx/kaggle-environments-0.1.4/kaggle_environments/status_codes.json
/kaggle/input/competitions/connectx/kaggle-environments-0.1.4/kaggle_environments/core.py
/kaggle/input/competitions/connectx/kaggle-environme

'\nenv.reset()\n# Play as the first agent against default "random" agent.\nenv.run([my_agent, "random"])\nenv.render(mode="ipython", width=500, height=450)\n'

In [2]:
##############################################
##############################################
#
#  PLAY A SINGLE GAME AGAINST AN ONLINE BOT
#
##############################################
##############################################


def play_game (player1, player2, bot_idx) :
    # Create a Connect X environment
    env = make("connectx", debug=True)
    
    # Run a single game
    env.run([player1, player2])

    reward = -1
    
    final_step = env.steps[-1]
    if final_step[0]['status'] == 'DONE':
        # Determine the result for your agent (player 1)
        # The reward for the agent is at index 0
        reward = final_step[bot_idx]['reward']
    else:
        reward = -1
        print("The game has not concluded yet.")
    
    # Render the final state of the board and the game replay
    # env.render(mode="ipython")

    return reward

In [3]:
##############################################
##############################################
#
#                 TESTER
#
##############################################
##############################################

NUM_GAMES = 50

# BOT PLAYS FIRST
bot_wins = 0
bot_losses = 0
draws = 0
test_start_time = time.time()
FIRST_GAMES = int(math.ceil(NUM_GAMES / 2))

print("BOT PLAYS FIRST")
for test_num in range (FIRST_GAMES) :
    print(f"Playing game {test_num+1} / {FIRST_GAMES}")
    reward = play_game(my_agent, "negamax", 0)

    if reward == 1:
        bot_wins += 1
        # print("BOT WON")
    elif reward == -1:
        bot_losses += 1
        # print("BOT LOST")
    else:  # reward == 0 or None
        draws += 1
        # print("DRAW")

print(f"WINS          = {bot_wins}")
print(f"LOSSES        = {bot_losses}")
print(f"DRAWS         = {draws}")
print(f"TOTAL GAMES   = {FIRST_GAMES}")
print(f"AVG TIME/GAME = {(time.time() - test_start_time)/FIRST_GAMES}")


# BOT PLAYS SECOND
bot_wins = 0
bot_losses = 0
draws = 0
test_start_time = time.time()
SECOND_GAMES = int(math.floor(NUM_GAMES / 2))

print("BOT PLAYS SECOND")
for test_num in range (SECOND_GAMES) :
    print(f"Playing game {test_num+1} / {SECOND_GAMES}")
    reward = play_game("negamax", my_agent, 1)

    if reward == 1:
        bot_wins += 1
        # print("BOT WON")
    elif reward == -1:
        bot_losses += 1
        # print("BOT LOST")
    else:  # reward == 0 or None
        draws += 1
        # print("DRAW")

print(f"WINS          = {bot_wins}")
print(f"LOSSES        = {bot_losses}")
print(f"DRAWS         = {draws}")
print(f"TOTAL GAMES   = {SECOND_GAMES}")
print(f"AVG TIME/GAME = {(time.time() - test_start_time)/SECOND_GAMES}")

BOT PLAYS FIRST
Playing game 1 / 25
Playing game 2 / 25
>2s TIMEOUT
Playing game 3 / 25
>2s TIMEOUT
Playing game 4 / 25
Playing game 5 / 25
Playing game 6 / 25
Playing game 7 / 25
>2s TIMEOUT
Playing game 8 / 25
Playing game 9 / 25
>2s TIMEOUT
Playing game 10 / 25
Playing game 11 / 25
>2s TIMEOUT
Playing game 12 / 25
Playing game 13 / 25
Playing game 14 / 25
>2s TIMEOUT
Playing game 15 / 25
Playing game 16 / 25
Playing game 17 / 25
Playing game 18 / 25
Timeout: Timed out after 5 seconds.>2s TIMEOUT

The game has not concluded yet.
Playing game 19 / 25
Playing game 20 / 25
Playing game 21 / 25
Playing game 22 / 25
Timeout: Timed out after 5 seconds.
The game has not concluded yet.
Playing game 23 / 25
>2s TIMEOUT
Playing game 24 / 25
Playing game 25 / 25
WINS          = 23
LOSSES        = 2
DRAWS         = 0
TOTAL GAMES   = 25
AVG TIME/GAME = 30.832010173797606
BOT PLAYS SECOND
Playing game 1 / 25
Playing game 2 / 25
Timeout: Timed out after 5 seconds.
Playing game 3 / 25
>2s TIMEOUT
Pl